# 01 - First look at the data

What the IBM Heron calibration archive contains for one backend, before any analysis.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

In [2]:
DATA = Path("../data")
RAW = DATA / "raw"
INTERIM = DATA / "interim"
PROCESSED = DATA / "processed"
RAW_FILE = RAW / "calibration.parquet"

In [3]:
DATASET = "phanerozoic/qiskit-calibration-drift"
REVISION = "9a6f6d6ff186cbebc66c5723f0bc02d9537bf011"  # 2026-09-27, 20:27
BACKEND = "ibm_fez"


## Load (downloaded once, then cached locally)

In [4]:
if RAW_FILE.exists():
    df = pd.read_parquet(RAW_FILE)
else:
    from datasets import load_dataset

    ds = load_dataset(DATASET, split="train", revision=REVISION)
    df = ds.to_pandas()
    RAW_FILE.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(RAW_FILE)

df.shape

(13452222, 31)

In [5]:
df

,backend,property_family,property,qubit_a,qubit_b,value,unit,scope,is_failure_ceiling,observed_time,...,bz_gsm_nt,neutron_flux,kp_index,ap_index,Ap_daily,SN,f107_observed_sfu,f107_adjusted_sfu,solar_flux_sfu,dst_nt
0,ibm_fez,T1,T1,0,NaN,4.853901e-05,NaN,NaN,False,2026-01-31 17:05:03.496046+00:00,...,-0.56,NaN,0.67,3.0,4.0,108.0,140.5,136.4,128.0,-6.0
1,ibm_fez,T1,T1,0,NaN,5.397765e-05,NaN,NaN,False,2026-02-08 08:44:35.389875+00:00,...,1.27,0.568,2.33,3.0,6.0,144.0,167.4,162.9,169.0,-3.0
2,ibm_fez,T1,T1,0,NaN,5.535806e-05,NaN,NaN,False,2026-02-09 09:48:42.736598+00:00,...,6.22,0.062,1.67,5.0,6.0,119.0,144.2,140.4,167.0,3.0
3,ibm_fez,T1,T1,0,NaN,5.744498e-05,NaN,NaN,False,2026-02-12 07:47:32.285992+00:00,...,-0.32,0.936,2.67,7.0,7.0,81.0,129.0,125.7,129.0,-11.0
4,ibm_fez,T1,T1,0,NaN,5.632442e-05,NaN,NaN,False,2026-02-14 10:42:58.934289+00:00,...,5.13,0.002,0.33,2.0,10.0,77.0,116.9,114.0,117.0,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13452217,ibm_kingston,measure_reset_2_threshold,measure_reset_2_threshold,153,NaN,4.897967e+06,,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN
13452218,ibm_kingston,measure_reset_2_gate_length,measure_reset_2_gate_length,154,NaN,1.412000e+03,ns,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN
13452219,ibm_kingston,measure_reset_2_threshold,measure_reset_2_threshold,154,NaN,-6.462877e+04,,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN
13452220,ibm_kingston,measure_reset_2_gate_length,measure_reset_2_gate_length,155,NaN,1.412000e+03,ns,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN


## Backends and time coverage

In [6]:
df.groupby("backend").agg(
    rows=("value", "size"),
    first=("observed_time", "min"),
    last=("observed_time", "max"),
    qubits=("qubit_a", "nunique"),
)

,rows,first,last,qubits
backend,,,,
ibm_fez,4599932,2026-01-31 17:05:03+00:00,2026-09-27 18:17:04+00:00,156
ibm_kingston,4017847,2026-03-16 12:50:43+00:00,2026-09-27 18:17:04+00:00,156
ibm_marrakesh,4453008,2026-01-31 17:05:03+00:00,2026-09-27 17:36:35+00:00,156
ibm_torino,381435,2026-01-31 17:05:03.496046+00:00,2026-04-01 14:43:42.453400+00:00,133


`ibm_torino` is discontinued, and `ibm_fez` is the richest one: we'll go with it

In [7]:
fez = df[df["backend"] == BACKEND].copy()

In [8]:
fez

,backend,property_family,property,qubit_a,qubit_b,value,unit,scope,is_failure_ceiling,observed_time,...,bz_gsm_nt,neutron_flux,kp_index,ap_index,Ap_daily,SN,f107_observed_sfu,f107_adjusted_sfu,solar_flux_sfu,dst_nt
0,ibm_fez,T1,T1,0,NaN,4.853901e-05,NaN,NaN,False,2026-01-31 17:05:03.496046+00:00,...,-0.56,NaN,0.67,3.0,4.0,108.0,140.5,136.4,128.0,-6.0
1,ibm_fez,T1,T1,0,NaN,5.397765e-05,NaN,NaN,False,2026-02-08 08:44:35.389875+00:00,...,1.27,0.568,2.33,3.0,6.0,144.0,167.4,162.9,169.0,-3.0
2,ibm_fez,T1,T1,0,NaN,5.535806e-05,NaN,NaN,False,2026-02-09 09:48:42.736598+00:00,...,6.22,0.062,1.67,5.0,6.0,119.0,144.2,140.4,167.0,3.0
3,ibm_fez,T1,T1,0,NaN,5.744498e-05,NaN,NaN,False,2026-02-12 07:47:32.285992+00:00,...,-0.32,0.936,2.67,7.0,7.0,81.0,129.0,125.7,129.0,-11.0
4,ibm_fez,T1,T1,0,NaN,5.632442e-05,NaN,NaN,False,2026-02-14 10:42:58.934289+00:00,...,5.13,0.002,0.33,2.0,10.0,77.0,116.9,114.0,117.0,-2.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13450275,ibm_fez,measure_reset_2_threshold,measure_reset_2_threshold,153,NaN,-2.326223e+06,,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN
13450276,ibm_fez,measure_reset_2_gate_length,measure_reset_2_gate_length,154,NaN,1.364000e+03,ns,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN
13450277,ibm_fez,measure_reset_2_threshold,measure_reset_2_threshold,154,NaN,-1.265037e+06,,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN
13450278,ibm_fez,measure_reset_2_gate_length,measure_reset_2_gate_length,155,NaN,1.364000e+03,ns,gate1q,False,2026-09-27 18:17:04+00:00,...,NaN,0.353,NaN,NaN,NaN,56.0,NaN,NaN,NaN,NaN
